# DataPilot AI — Priority 6: QLoRA fine-tuning (Colab T4)\n\n**Model:** `Qwen/Qwen2.5-1.5B-Instruct`  \n**Data:** Dataset B (`data/finetuning/train.jsonl`, `validation.jsonl`)  \n**Output:** LoRA adapter under `experiments/results/adapters/<run>/adapter/`\n\nRuntime → **GPU** → **T4**. Do not invent metrics; use only `training_summary.json` from the run.

## 1. Setup\n\nUpload/clone the project so `config/`, `data/finetuning/`, and `src/` are available, then install deps.

In [ ]:
# Option A: mount Google Drive and point at your project folder
from google.colab import drive
drive.mount("/content/drive")

# EDIT this path to your uploaded Masters_Project folder on Drive
PROJECT_DIR = "/content/drive/MyDrive/Masters_Project"

import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print("cwd:", os.getcwd())

In [ ]:
# Colab 2026: PyTorch cu128 has no bitsandbytes GPU wheel (triton.ops crash).
# Qwen 1.5B fits T4 in fp16 — skip bitsandbytes entirely.
%pip uninstall -y bitsandbytes torchao
%pip -q install -U peft trl accelerate datasets pyyaml python-dotenv

import torch
assert torch.cuda.is_available(), "Enable GPU runtime (T4) before training"
print("torch", torch.__version__)
print(torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 2. Dry-run (format check)

In [ ]:
!python scripts/train_qlora.py --dry-run

## 3. Train (fp16 LoRA on current Colab)\n\nUses `config/model.yaml` (3 epochs default, early stopping patience 2).  \n**`--no-4bit` is required** on Colab CUDA 12.8. Qwen 1.5B still fits T4.  \nFor a quick GPU smoke test first, use the commented command with sample caps.

In [ ]:
# Smoke test (optional):
# !python scripts/train_qlora.py --max-train-samples 32 --max-val-samples 8 --run-name colab_smoke_qlora --no-4bit -v

# Full Dataset B training (fp16 LoRA — skip bitsandbytes):
!python scripts/train_qlora.py --run-name colab_t4_qlora_v1 --no-4bit -v

## 4. Inspect real metrics\n\nOnly numbers written by Trainer are valid for the thesis.

In [ ]:
from pathlib import Path
import json

runs = sorted(Path("experiments/results/adapters").glob("*/training_summary.json"))
assert runs, "No training_summary.json found — training did not finish"
summary = json.loads(runs[-1].read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2))
print("\\nAdapter path:", summary.get("adapter_path"))

## 5. Next (Priority 7)\n\nCopy the `adapter/` folder back into the project (already on Drive if you trained there), then wire FT+RAG with:\n\n```python\nHuggingFaceLLM.from_configs(adapter_path="experiments/results/adapters/<run>/adapter")\n```\n\nDo not claim RAG+FT quality gains until Priority 9–10 evaluation runs complete.